# Sentiment Analysis of Hotel Reviews Using Deep Learning Models

This notebook implements the proposed project: classifying TripAdvisor hotel reviews into **positive**, **neutral**, and **negative** sentiment categories using traditional machine learning, LSTM, and transformer-based models.

## Project alignment

**Problem:** Online hotel reviews contain valuable customer satisfaction insights, but the high volume and unstructured text format make manual analysis slow and inconsistent.

**Input:** Hotel review text and numerical rating score.  
**Output:** Sentiment label: negative, neutral, or positive.  
**Primary success metric:** Macro F1-score, because the neutral and negative classes may be smaller than the positive class.  
**Secondary metrics:** Accuracy, weighted F1-score, precision, recall, confusion matrix, and per-class performance.

## Models compared

1. **Baseline:** TF-IDF + Logistic Regression
2. **Deep learning:** Embedding + BiLSTM
3. **Transformer:** DistilBERT / BERT sequence classifier
4. **Controlled comparison / ablation:** TF-IDF feature setting comparison and optional LSTM sequence-length comparison

> The notebook is designed to run end-to-end once the Kaggle dataset CSV is available locally.

In [ ]:
# If needed, install dependencies in your environment first.
# Uncomment only when running in a fresh notebook environment.
# !pip install -q pandas numpy scikit-learn matplotlib seaborn tensorflow transformers datasets accelerate kagglehub

In [ ]:
import os
import re
import json
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)
from sklearn.utils.class_weight import compute_class_weight

warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

LABEL_MAP = {0: 'negative', 1: 'neutral', 2: 'positive'}
LABEL_TO_ID = {v: k for k, v in LABEL_MAP.items()}

## 1. Load the TripAdvisor hotel reviews dataset

Dataset source: Kaggle dataset `thedevastator/tripadvisor-hotel-reviews`.

Expected columns usually include review text and rating score. The loader below is flexible and tries common column names such as `Review`, `review`, `text`, `Rating`, `rating`, and `score`.

In [ ]:
def find_dataset_csv(search_root='.'):
    """Find a likely hotel review CSV file in the current directory tree."""
    search_root = Path(search_root)
    candidates = list(search_root.rglob('*.csv'))
    if not candidates:
        return None
    # Prefer files with hotel/tripadvisor/review in the name
    scored = []
    for p in candidates:
        name = p.name.lower()
        score = sum(term in name for term in ['hotel', 'tripadvisor', 'review'])
        scored.append((score, p))
    scored.sort(reverse=True, key=lambda x: x[0])
    return scored[0][1]

# Option A: Manually set your downloaded Kaggle CSV path here.
DATA_PATH = None  # Example: 'tripadvisor_hotel_reviews.csv'

# Option B: Auto-detect CSV in notebook folder.
if DATA_PATH is None:
    detected = find_dataset_csv('.')
    if detected is not None:
        DATA_PATH = str(detected)

# Option C: Try KaggleHub download if no local CSV is found.
if DATA_PATH is None:
    try:
        import kagglehub
        dataset_dir = kagglehub.dataset_download('thedevastator/tripadvisor-hotel-reviews')
        detected = find_dataset_csv(dataset_dir)
        if detected is not None:
            DATA_PATH = str(detected)
    except Exception as e:
        print('KaggleHub download was not available:', e)

if DATA_PATH is None:
    raise FileNotFoundError(
        'Dataset CSV not found. Download the Kaggle dataset and place the CSV in the same folder as this notebook, '
        'or set DATA_PATH manually.'
    )

print('Using dataset:', DATA_PATH)
df_raw = pd.read_csv(DATA_PATH)
print(df_raw.shape)
df_raw.head()

In [ ]:
def infer_columns(df):
    text_candidates = ['Review', 'review', 'reviews', 'text', 'Text', 'content', 'Content']
    rating_candidates = ['Rating', 'rating', 'ratings', 'score', 'Score', 'stars', 'Stars']

    text_col = next((c for c in text_candidates if c in df.columns), None)
    rating_col = next((c for c in rating_candidates if c in df.columns), None)

    if text_col is None:
        # Fallback: choose the object column with the longest average text
        object_cols = df.select_dtypes(include=['object']).columns
        if len(object_cols) > 0:
            text_col = max(object_cols, key=lambda c: df[c].astype(str).str.len().mean())

    if rating_col is None:
        # Fallback: choose a numeric column whose values mostly lie between 1 and 5
        numeric_cols = df.select_dtypes(include=[np.number]).columns
        possible = []
        for c in numeric_cols:
            vals = df[c].dropna()
            if len(vals) and vals.between(1, 5).mean() > 0.9:
                possible.append(c)
        rating_col = possible[0] if possible else None

    if text_col is None or rating_col is None:
        raise ValueError(f'Could not infer text/rating columns. Columns found: {list(df.columns)}')
    return text_col, rating_col

text_col, rating_col = infer_columns(df_raw)
print('Text column:', text_col)
print('Rating column:', rating_col)

df = df_raw[[text_col, rating_col]].rename(columns={text_col: 'review', rating_col: 'rating'}).copy()
df = df.dropna(subset=['review', 'rating'])
df['review'] = df['review'].astype(str)
df['rating'] = pd.to_numeric(df['rating'], errors='coerce')
df = df.dropna(subset=['rating'])
df = df[df['rating'].between(1, 5)]
df['rating'] = df['rating'].round().astype(int)

def rating_to_sentiment_id(rating):
    if rating <= 2:
        return 0  # negative
    if rating == 3:
        return 1  # neutral
    return 2      # positive

df['label'] = df['rating'].apply(rating_to_sentiment_id)
df['sentiment'] = df['label'].map(LABEL_MAP)

df.head()

## 2. Exploratory analysis and class distribution

This step checks whether the dataset is imbalanced. Imbalance is expected because hotel review datasets often contain many positive reviews. Macro F1 is therefore emphasized during evaluation.

In [ ]:
print(df.info())
print('
Class distribution:')
print(df['sentiment'].value_counts())
print('
Class percentage:')
print((df['sentiment'].value_counts(normalize=True) * 100).round(2))

plt.figure(figsize=(7, 4))
df['sentiment'].value_counts().reindex(['negative', 'neutral', 'positive']).plot(kind='bar')
plt.title('Sentiment Class Distribution')
plt.xlabel('Sentiment')
plt.ylabel('Number of Reviews')
plt.xticks(rotation=0)
plt.show()

review_lengths = df['review'].str.split().str.len()
print(review_lengths.describe())
plt.figure(figsize=(7, 4))
plt.hist(review_lengths, bins=40)
plt.title('Review Length Distribution')
plt.xlabel('Words per Review')
plt.ylabel('Frequency')
plt.show()

## 3. Preprocessing and stratified train/validation/test split

The test set is held out until final evaluation. The validation set is used for model selection and early stopping.

In [ ]:
def clean_text(text):
    """Light cleaning for TF-IDF and LSTM models. Transformer models use their own tokenizer."""
    text = str(text).lower()
    text = re.sub(r'<.*?>', ' ', text)
    text = re.sub(r'http\S+|www\.\S+', ' ', text)
    text = re.sub(r'[^a-z0-9\s'\-]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['clean_review'] = df['review'].apply(clean_text)

X_temp, X_test, y_temp, y_test = train_test_split(
    df['clean_review'], df['label'], test_size=0.15, random_state=SEED, stratify=df['label']
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.1765, random_state=SEED, stratify=y_temp
)

# Approximately 70% train, 15% validation, 15% test
print('Train:', X_train.shape, 'Validation:', X_val.shape, 'Test:', X_test.shape)
print('Train distribution:', y_train.map(LABEL_MAP).value_counts(normalize=True).round(3).to_dict())
print('Validation distribution:', y_val.map(LABEL_MAP).value_counts(normalize=True).round(3).to_dict())
print('Test distribution:', y_test.map(LABEL_MAP).value_counts(normalize=True).round(3).to_dict())

## 4. Shared evaluation utilities

In [ ]:
results = []

def evaluate_model(name, y_true, y_pred, show_report=True, show_matrix=True):
    acc = accuracy_score(y_true, y_pred)
    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
        y_true, y_pred, average='macro', zero_division=0
    )
    precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(
        y_true, y_pred, average='weighted', zero_division=0
    )
    row = {
        'model': name,
        'accuracy': acc,
        'macro_precision': precision_macro,
        'macro_recall': recall_macro,
        'macro_f1': f1_macro,
        'weighted_f1': f1_weighted,
    }
    results.append(row)

    print(f'
{name}')
    print('-' * len(name))
    print(pd.Series(row).drop('model').round(4))
    if show_report:
        print('
Classification report:')
        print(classification_report(y_true, y_pred, target_names=[LABEL_MAP[i] for i in sorted(LABEL_MAP)], zero_division=0))
    if show_matrix:
        cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2])
        disp = ConfusionMatrixDisplay(cm, display_labels=[LABEL_MAP[i] for i in [0, 1, 2]])
        disp.plot(values_format='d')
        plt.title(f'Confusion Matrix: {name}')
        plt.show()
    return row

## 5. Baseline 1: Majority-class classifier

This sanity-check baseline predicts the most frequent class. A useful model should beat this baseline clearly, especially on macro F1.

In [ ]:
dummy = DummyClassifier(strategy='most_frequent', random_state=SEED)
dummy.fit(X_train, y_train)
y_pred_dummy = dummy.predict(X_test)
evaluate_model('Majority Class Baseline', y_test, y_pred_dummy)

## 6. Baseline 2: TF-IDF + Logistic Regression

This is the main traditional machine learning baseline. TF-IDF represents reviews as weighted word and phrase features, while Logistic Regression learns a linear decision boundary.

In [ ]:
tfidf_lr = Pipeline([
    ('tfidf', TfidfVectorizer(
        max_features=20000,
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.95,
        sublinear_tf=True
    )),
    ('clf', LogisticRegression(
        max_iter=2000,
        class_weight='balanced',
        solver='liblinear',
        random_state=SEED
    ))
])

tfidf_lr.fit(X_train, y_train)
y_pred_lr = tfidf_lr.predict(X_test)
evaluate_model('TF-IDF + Logistic Regression', y_test, y_pred_lr)

## 7. Controlled comparison / ablation: TF-IDF feature design

This ablation tests whether adding bigrams improves sentiment classification compared with unigram-only features. It isolates the effect of phrase-level features such as “not clean”, “very good”, or “front desk”.

In [ ]:
ablation_settings = {
    'TF-IDF unigram only': (1, 1),
    'TF-IDF unigram + bigram': (1, 2),
}

for name, ngram_range in ablation_settings.items():
    model = Pipeline([
        ('tfidf', TfidfVectorizer(max_features=20000, ngram_range=ngram_range, min_df=2, max_df=0.95, sublinear_tf=True)),
        ('clf', LogisticRegression(max_iter=2000, class_weight='balanced', solver='liblinear', random_state=SEED))
    ])
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    evaluate_model(name, y_test, preds, show_report=False, show_matrix=False)

pd.DataFrame(results).sort_values('macro_f1', ascending=False).round(4)

## 8. Deep learning model: Embedding + Bidirectional LSTM

The LSTM model is appropriate because hotel reviews are sequences of words. A bidirectional LSTM can use both earlier and later words to capture contextual patterns that a bag-of-words model may miss.

In [ ]:
try:
    import tensorflow as tf
    from tensorflow.keras.preprocessing.text import Tokenizer
    from tensorflow.keras.preprocessing.sequence import pad_sequences
    from tensorflow.keras import layers, models, callbacks

    tf.random.set_seed(SEED)

    MAX_WORDS = 20000
    MAX_LEN = 220
    EMBEDDING_DIM = 128
    BATCH_SIZE = 32
    EPOCHS = 10

    tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token='<OOV>')
    tokenizer.fit_on_texts(X_train)

    def encode_lstm(texts, max_len=MAX_LEN):
        seq = tokenizer.texts_to_sequences(texts)
        return pad_sequences(seq, maxlen=max_len, padding='post', truncating='post')

    X_train_seq = encode_lstm(X_train)
    X_val_seq = encode_lstm(X_val)
    X_test_seq = encode_lstm(X_test)

    y_train_np = y_train.to_numpy()
    y_val_np = y_val.to_numpy()
    y_test_np = y_test.to_numpy()

    classes = np.array([0, 1, 2])
    weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_train_np)
    class_weight = dict(zip(classes, weights))
    print('Class weights:', class_weight)

    lstm_model = models.Sequential([
        layers.Embedding(input_dim=MAX_WORDS, output_dim=EMBEDDING_DIM, input_length=MAX_LEN),
        layers.SpatialDropout1D(0.25),
        layers.Bidirectional(layers.LSTM(64, return_sequences=False, dropout=0.25, recurrent_dropout=0.0)),
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.4),
        layers.Dense(3, activation='softmax')
    ])

    lstm_model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    lstm_model.summary()

    early_stop = callbacks.EarlyStopping(
        monitor='val_loss', patience=2, restore_best_weights=True
    )

    history = lstm_model.fit(
        X_train_seq,
        y_train_np,
        validation_data=(X_val_seq, y_val_np),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        class_weight=class_weight,
        callbacks=[early_stop],
        verbose=1
    )

    plt.figure(figsize=(7, 4))
    plt.plot(history.history['loss'], label='train_loss')
    plt.plot(history.history['val_loss'], label='val_loss')
    plt.title('LSTM Training and Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.show()

    y_pred_lstm = np.argmax(lstm_model.predict(X_test_seq), axis=1)
    evaluate_model('Embedding + BiLSTM', y_test_np, y_pred_lstm)

except Exception as e:
    print('TensorFlow/LSTM section skipped because of this error:')
    print(e)

## 9. Transformer model: DistilBERT sequence classifier

DistilBERT uses contextual embeddings and self-attention. This model directly tests the research question: whether transformer-based architectures improve hotel review sentiment classification over traditional ML and LSTM models.

To keep compute manageable, the default settings use a small number of epochs and a maximum sequence length of 256. Increase these settings only if GPU resources are available.

In [ ]:
RUN_TRANSFORMER = True  # Set to False if running on CPU or if training time is limited.

try:
    if RUN_TRANSFORMER:
        import torch
        from datasets import Dataset
        from transformers import (
            AutoTokenizer,
            AutoModelForSequenceClassification,
            TrainingArguments,
            Trainer,
            DataCollatorWithPadding,
        )

        MODEL_NAME = 'distilbert-base-uncased'
        MAX_TRANSFORMER_LEN = 256

        tokenizer_tr = AutoTokenizer.from_pretrained(MODEL_NAME)

        train_df = pd.DataFrame({'text': X_train.tolist(), 'label': y_train.tolist()})
        val_df = pd.DataFrame({'text': X_val.tolist(), 'label': y_val.tolist()})
        test_df = pd.DataFrame({'text': X_test.tolist(), 'label': y_test.tolist()})

        train_ds = Dataset.from_pandas(train_df)
        val_ds = Dataset.from_pandas(val_df)
        test_ds = Dataset.from_pandas(test_df)

        def tokenize_batch(batch):
            return tokenizer_tr(batch['text'], truncation=True, max_length=MAX_TRANSFORMER_LEN)

        train_ds = train_ds.map(tokenize_batch, batched=True)
        val_ds = val_ds.map(tokenize_batch, batched=True)
        test_ds = test_ds.map(tokenize_batch, batched=True)

        id2label = LABEL_MAP
        label2id = LABEL_TO_ID

        transformer_model = AutoModelForSequenceClassification.from_pretrained(
            MODEL_NAME,
            num_labels=3,
            id2label=id2label,
            label2id=label2id
        )

        def compute_metrics(eval_pred):
            logits, labels = eval_pred
            preds = np.argmax(logits, axis=-1)
            acc = accuracy_score(labels, preds)
            p, r, f1, _ = precision_recall_fscore_support(labels, preds, average='macro', zero_division=0)
            return {'accuracy': acc, 'macro_precision': p, 'macro_recall': r, 'macro_f1': f1}

        training_args = TrainingArguments(
            output_dir='./hotel_sentiment_distilbert',
            learning_rate=2e-5,
            per_device_train_batch_size=8,
            per_device_eval_batch_size=16,
            num_train_epochs=2,
            weight_decay=0.01,
            eval_strategy='epoch',
            save_strategy='epoch',
            load_best_model_at_end=True,
            metric_for_best_model='macro_f1',
            greater_is_better=True,
            logging_steps=50,
            seed=SEED,
            report_to=[]
        )

        trainer = Trainer(
            model=transformer_model,
            args=training_args,
            train_dataset=train_ds,
            eval_dataset=val_ds,
            tokenizer=tokenizer_tr,
            data_collator=DataCollatorWithPadding(tokenizer=tokenizer_tr),
            compute_metrics=compute_metrics,
        )

        trainer.train()
        transformer_predictions = trainer.predict(test_ds)
        y_pred_transformer = np.argmax(transformer_predictions.predictions, axis=1)
        evaluate_model('DistilBERT Transformer', y_test.to_numpy(), y_pred_transformer)
    else:
        print('Transformer training skipped. Set RUN_TRANSFORMER=True to run it.')
except Exception as e:
    print('Transformer section skipped because of this error:')
    print(e)

## 10. Final results comparison

Interpret the table using **macro F1** as the primary metric. Accuracy can be misleading if the positive class dominates the dataset.

In [ ]:
results_df = pd.DataFrame(results).drop_duplicates(subset=['model'], keep='last')
results_df = results_df.sort_values('macro_f1', ascending=False).reset_index(drop=True)
results_df.round(4)

In [ ]:
if not results_df.empty:
    plt.figure(figsize=(9, 4))
    plt.bar(results_df['model'], results_df['macro_f1'])
    plt.title('Model Comparison by Macro F1-score')
    plt.xlabel('Model')
    plt.ylabel('Macro F1-score')
    plt.xticks(rotation=35, ha='right')
    plt.ylim(0, 1)
    plt.show()

## 11. Failure analysis

This section identifies examples where the best available model is wrong. Typical sentiment-analysis errors may occur when reviews contain mixed opinions, sarcasm, short neutral comments, or ratings that do not fully match the text.

In [ ]:
# Choose the best model available for failure analysis.
# Priority: Transformer if available, then LSTM, then TF-IDF Logistic Regression.
if 'y_pred_transformer' in globals():
    best_model_name = 'DistilBERT Transformer'
    best_preds = y_pred_transformer
elif 'y_pred_lstm' in globals():
    best_model_name = 'Embedding + BiLSTM'
    best_preds = y_pred_lstm
else:
    best_model_name = 'TF-IDF + Logistic Regression'
    best_preds = y_pred_lr

failure_df = pd.DataFrame({
    'review': X_test.reset_index(drop=True),
    'true_label': pd.Series(y_test).reset_index(drop=True).map(LABEL_MAP),
    'predicted_label': pd.Series(best_preds).map(LABEL_MAP),
})
failure_df['is_error'] = failure_df['true_label'] != failure_df['predicted_label']
errors = failure_df[failure_df['is_error']].copy()

print(f'Best model used for failure analysis: {best_model_name}')
print(f'Number of test errors: {len(errors)} out of {len(failure_df)}')

# Show a compact sample of errors for qualitative analysis.
pd.set_option('display.max_colwidth', 220)
errors[['true_label', 'predicted_label', 'review']].head(15)

In [ ]:
# Error type breakdown
if len(errors) > 0:
    error_breakdown = errors.groupby(['true_label', 'predicted_label']).size().reset_index(name='count')
    display(error_breakdown.sort_values('count', ascending=False))

    print('
Interpretation prompts for report writing:')
    print('- Are neutral reviews often confused with positive or negative?')
    print('- Do mixed reviews mention both praise and complaints?')
    print('- Are short reviews harder than long reviews?')
    print('- Are rating-derived labels sometimes noisy because text and score disagree?')

## 12. Inference demo

Use this function to classify new hotel reviews. The function uses the best trained model available in the current notebook session.

In [ ]:
def predict_sentiment(review_text):
    review_clean = clean_text(review_text)
    if 'trainer' in globals() and 'tokenizer_tr' in globals():
        # Transformer inference
        import torch
        inputs = tokenizer_tr(review_clean, return_tensors='pt', truncation=True, max_length=MAX_TRANSFORMER_LEN)
        device = next(transformer_model.parameters()).device
        inputs = {k: v.to(device) for k, v in inputs.items()}
        transformer_model.eval()
        with torch.no_grad():
            logits = transformer_model(**inputs).logits
        pred_id = int(torch.argmax(logits, dim=-1).cpu().numpy()[0])
        return LABEL_MAP[pred_id]
    elif 'lstm_model' in globals():
        seq = encode_lstm([review_clean])
        pred_id = int(np.argmax(lstm_model.predict(seq, verbose=0), axis=1)[0])
        return LABEL_MAP[pred_id]
    else:
        pred_id = int(tfidf_lr.predict([review_clean])[0])
        return LABEL_MAP[pred_id]

sample_reviews = [
    'The room was spotless, the staff were friendly, and the breakfast was excellent.',
    'The location was good, but the room was noisy and the bathroom was not clean.',
    'Average stay. Nothing special, but it was acceptable for one night.'
]

for r in sample_reviews:
    print(f'Review: {r}')
    print(f'Predicted sentiment: {predict_sentiment(r)}
')

## 13. Ethics, limitations, and responsible use

- **Bias:** Public online reviews may overrepresent guests with very positive or very negative experiences.
- **Label noise:** Ratings are used as sentiment labels, but rating score and review text can disagree.
- **Privacy:** The project uses publicly available review text only. No private user data or API keys are required.
- **Interpretability:** Deep models may improve performance but are harder to interpret than TF-IDF Logistic Regression.
- **Responsible use:** Predictions should support human decision-making, not replace customer service judgment.

## Report-ready conclusion template

Replace bracketed values after running the notebook:

> The best-performing model was **[model name]**, achieving a macro F1-score of **[score]** on the held-out test set. Compared with the TF-IDF Logistic Regression baseline, the deep learning / transformer model **[improved/did not improve]** performance. Error analysis showed that the most difficult cases were **[neutral/mixed/short/noisy]** reviews, suggesting that hotel sentiment classification is affected by class imbalance, mixed opinions, and rating-derived label noise.